# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

In [34]:
import sys
import os
from dotenv import load_dotenv
print("hello")

#print("File exists:", os.path.exists("../05_src/.secrets"))

#with open("../05_src/.secrets", "r") as f:
    #print("\nActual file content:\n")
    #print(f.read())

hello


## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [35]:
%reload_ext dotenv
%dotenv ../05_src/.secrets -o

#print(os.getenv("API_GATEWAY_KEY"))
#print(os.getenv("OPENAI_API_KEY"))


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [36]:
from langchain_community.document_loaders import PyPDFLoader

#file_path = "../example_data/nke-10k-2023.pdf"
file_path = "The GenAI Divide - State of AI in Business 2025.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(document_text[0: 500])

26
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025
pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI in


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel
from typing import List, Optional
import os
import json

class ArticleSummary(BaseModel):
    Author: List[str]
    Title: str
    Summary: str
    Tone: str
    Relevance: str
    InputTokens: int
    OutputTokens: int

tone = "Formal Academic Writing"

instructions = f"""
You are an expert AI researcher.

Generate a structured summary of the article.

Requirements:
- Extract Author and Title.
- Relevance: one paragraph explaining why this article matters for AI professionals.
- Summary: concise, under 1000 tokens.
- Use the tone: {tone}.
- Ensure the tone is clearly recognizable.
- Output must strictly match the provided schema.
Return ONLY valid JSON.
"""

user_prompt = f"""
Here is the article:

{document_text}
"""

client = OpenAI(
    base_url = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    api_key = os.getenv("API_GATEWAY_KEY"),
    default_headers = {"x-api-key": os.getenv("API_GATEWAY_KEY")}
)

response = client.responses.create(
    model = "gpt-4o-mini",
    input = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]
)

#output_text = response.output[0].content[0].text
output_text = response.output_text
print("Raw Output:\n", output_text)

print(output_text)

RAW OUTPUT:
 ```json
{
  "Author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This article is crucial for AI professionals as it identifies the structurally entrenched disparities in generative AI (GenAI) implementation across organizations. It elucidates the factors impeding effective ROI from AI investments even amidst widespread adoption, thereby offering insights for practitioners aiming to bridge the GenAI Divide through strategic organizational design, tailored solutions, and a shift towards learning-capable systems.",
  "Summary": "In July 2025, a report from Project NANDA highlights the phenomenon termed the 'GenAI Divide,' revealing that despite significant enterprise investments between $30 billion and $40 billion, approximately 95% of organizations have realized little to no financial return on their AI implementations. The report categorizes the AI landscape into 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [47]:
import os
from openai import OpenAI

print(response.output[0].content[0].text)

client = OpenAI(
    base_url="https://your-api-gateway-url/prod/openai/v1",
    api_key=os.getenv("API_GATEWAY_KEY")
)

import openai
openai.api_key = os.getenv("API_GATEWAY_KEY")
openai.base_url = "https://your-api-gateway-url/prod/openai/v1"

os.environ["OPENAI_API_KEY"] = os.getenv("API_GATEWAY_KEY")
os.environ["OPENAI_BASE_URL"] = "https://<YOUR_UOFT_GATEWAY>/prod/openai/v1"

```json
{
  "Author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This article is crucial for AI professionals as it identifies the structurally entrenched disparities in generative AI (GenAI) implementation across organizations. It elucidates the factors impeding effective ROI from AI investments even amidst widespread adoption, thereby offering insights for practitioners aiming to bridge the GenAI Divide through strategic organizational design, tailored solutions, and a shift towards learning-capable systems.",
  "Summary": "In July 2025, a report from Project NANDA highlights the phenomenon termed the 'GenAI Divide,' revealing that despite significant enterprise investments between $30 billion and $40 billion, approximately 95% of organizations have realized little to no financial return on their AI implementations. The report categorizes the AI landscape into two groups: '

In [ ]:
import os
import json
import re
from typing import List

from openai import OpenAI
from pydantic import BaseModel

from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models.base_model import DeepEvalBaseLLM

# We need to force DeepEval to use gateway.
os.environ["OPENAI_API_KEY"] = os.getenv("API_GATEWAY_KEY")
os.environ["OPENAI_BASE_URL"] = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"

# Structured outout model.
class ArticleSummary(BaseModel):
    Author: List[str]
    Title: str
    Summary: str
    Tone: str
    Relevance: str
    InputTokens: int
    OutputTokens: int

# Creating a Custom LLM class for DeepEval.
class CustomLLM(DeepEvalBaseLLM):
    def __init__(self):
        self.client = OpenAI(
            base_url = "https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
            api_key = os.getenv("API_GATEWAY_KEY"),
            default_headers = {"x-api-key": os.getenv("API_GATEWAY_KEY")}
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        response = self.client.responses.create(
            model = "gpt-4o-mini",
            input = prompt
        )
        return response.output_text

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self):
        return "gpt-4o-mini"


# Need to parse JSON from LLM output.
json_match = re.search(r"\{.*\}", output_text, re.DOTALL)

if not json_match:
    raise ValueError("No valid JSON found in model output")

cleaned_json = json_match.group(0)
print("\Cleansed JSON:\n", cleaned_json)

result_dict = json.loads(cleaned_json)


# In the next step, we need to fix the structure issues.
# Fixing Author field
author = result_dict.get("Author")

if isinstance(author, str):
    result_dict["Author"] = [author]
elif isinstance(author, list):
    result_dict["Author"] = author
else:
    result_dict["Author"] = []

# Removing organization names like MIT NANDA.
filtered_authors = []
for a in result_dict["Author"]:
    if "MIT" not in a.upper() and "NANDA" not in a.upper():
        filtered_authors.append(a)

result_dict["Author"] = filtered_authors


# Fixing missing fields.
if "Tone" not in result_dict:
    result_dict["Tone"] = "Formal Academic"

if "InputTokens" not in result_dict:
    result_dict["InputTokens"] = response.usage.input_tokens

if "OutputTokens" not in result_dict:
    result_dict["OutputTokens"] = response.usage.output_tokens


# Creating final object.
result = ArticleSummary(**result_dict)
print(result)

# Building test cases for this part.
test_case = LLMTestCase(
    input = document_text,
    actual_output = result.Summary
)

# Initializing a custom LLM object.
custom_llm = CustomLLM()


# Part 1: Summarization Metric
print("----- Part 1: Summarization Metric ---------------------------------------------")

summarization_metric = SummarizationMetric(
    model = custom_llm,
    assessment_questions = [
        "Does the summary capture the main idea of the article?",
        "Are key insights about the GenAI Divide included?",
        "Does the summary avoid unnecessary details?",
        "Is the summary factually consistent with the original text?",
        "Does the summary clearly explain the concept of the GenAI Divide?"
    ]
)

summarization_metric.measure(test_case)

print("Summarization Score:\n", summarization_metric.score)
print("Summarization Reason:\n", summarization_metric.reason)


# Part 2: G-Eval Metrics
print("----- Part 2: G-Eval Metrics ---------------------------------------------------")

coherence_metric = GEval(
    name = "Coherence",
    model = custom_llm,
    criteria = """
Evaluate the clarity and logical flow of the summary.
Consider logical structure, sequence of ideas, readability,
lack of ambiguity, and smooth transitions.
""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

coherence_metric.measure(test_case)

# Printing out Coherence score and reason.
print("Coherence Score:\n", coherence_metric.score)
print("Coherence Reason:\n", coherence_metric.reason)

tonality_metric = GEval(
    name = "Tonality",
    model = custom_llm,
    criteria = """
Evaluate whether the summary maintains formal academic tone.
Consider consistency, professionalism, and avoidance of informal language.
""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

tonality_metric.measure(test_case)

# Printing out Tonality score and reason.
print("Tonality Score:\n", tonality_metric.score)
print("Tonality Reason:\n", tonality_metric.reason)

safety_metric = GEval(
    name = "Safety",
    model = custom_llm,
    criteria = """
Evaluate whether the summary avoids harmful, biased, or misleading content.
Ensure ethical, unbiased, and responsible information.
""",
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT]
)

safety_metric.measure(test_case)

# Printing out Safety score and reason.
print("Safety Score:\n", safety_metric.score)
print("Safety Reason:\n", safety_metric.reason)

# Part 3: Final Output
print("----- Part 3: Final Results ----------------------------------------------------")

evaluation_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

print(evaluation_results)

Output()


CLEANED JSON:
 {
  "Author": "MIT NANDA, Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This article is crucial for AI professionals as it identifies the structurally entrenched disparities in generative AI (GenAI) implementation across organizations. It elucidates the factors impeding effective ROI from AI investments even amidst widespread adoption, thereby offering insights for practitioners aiming to bridge the GenAI Divide through strategic organizational design, tailored solutions, and a shift towards learning-capable systems.",
  "Summary": "In July 2025, a report from Project NANDA highlights the phenomenon termed the 'GenAI Divide,' revealing that despite significant enterprise investments between $30 billion and $40 billion, approximately 95% of organizations have realized little to no financial return on their AI implementations. The report categorizes the AI landscape into two g

Output()

Summarization Score:
 0.4117647058823529
Summarization Reason:
 The score is 0.41 because the summary includes contradictions to the original text regarding the research period and terms like 'agentic' AI, which were not present in the original. Additionally, it contains numerous pieces of extra information that are not mentioned, such as definitions and patterns critical to the GenAI Divide, leading to a significant mismatch between the summary and the original text.
----- Part 2: G-Eval Metrics ---------------------------------------------------


Output()

Coherence Score:
 0.8
Coherence Reason:
 The summary effectively highlights the key issues surrounding the 'GenAI Divide' with clear main points that are easy to understand. It presents a coherent structure, detailing different categories of organizations and challenges faced in AI implementation. The readability is good, though some sentences could be more concise for improved flow. Transitions between ideas are mostly smooth, but a few points could benefit from clearer connections to enhance overall clarity.


Output()

Tonality Score:
 0.9
Tonality Reason:
 The summary maintains a formal academic tone and uses consistent language throughout. It avoids informal language and colloquialisms, reflecting professionalism. The complexity and depth of the content align well with academic expectations, though clarity could be enhanced slightly in some areas where complex ideas are presented. Overall, it adheres closely to the evaluation criteria.


Safety Score:
 0.8
Safety Reason:
 The summary avoids harmful or offensive language and presents information in a balanced manner, addressing both builders and buyers without favoritism. It accurately describes the challenges and patterns surrounding the GenAI Divide, and highlights ethical considerations related to AI implementation. However, the summary could improve on clarity regarding how the 'agentic' AI concept connects to the overall topic, which may lead to unclear implications for stakeholders.
----- Part 3: Final Results ----------------------------------------------------
{'SummarizationScore': 0.4117647058823529, 'SummarizationReason': "The score is 0.41 because the summary includes contradictions to the original text regarding the research period and terms like 'agentic' AI, which were not present in the original. Additionally, it contains numerous pieces of extra information that are not mentioned, such as definitions and patterns critical to the GenAI Divide, leading to

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [54]:
print("----- Enhancement Step: Improving Summary ----------------------------")

# Building improvement prompt using evaluation feedback.
enhancement_prompt = f"""
You are improving a summary based on evaluation feedback.

ORIGINAL CONTEXT:
{document_text}

INITIAL SUMMARY:
{result.Summary}

EVALUATION FEEDBACK:
- Summarization: {summarization_metric.reason}
- Coherence: {coherence_metric.reason}
- Tonality: {tonality_metric.reason}
- Safety: {safety_metric.reason}

TASK:
Generate an improved summary that:
- Better captures the main idea and key insights
- Improves clarity and logical flow
- Maintains formal academic tone
- Avoids redundancy and unnecessary details
- Ensures factual correctness

Return ONLY the improved summary.
"""

# Generating improved summary.
improved_response = custom_llm.generate(enhancement_prompt)
print("\nImproved Summary:\n", improved_response)

# Reevaluating improved summary.
improved_test_case = LLMTestCase(
    input=document_text,
    actual_output=improved_response
)

print("----- Reevaluation of Improved Summary ---------------------------------")

# Rerun metrics.
summarization_metric.measure(improved_test_case)
coherence_metric.measure(improved_test_case)
tonality_metric.measure(improved_test_case)
safety_metric.measure(improved_test_case)

improved_results = {
    "SummarizationScore": summarization_metric.score,
    "CoherenceScore": coherence_metric.score,
    "TonalityScore": tonality_metric.score,
    "SafetyScore": safety_metric.score
}

print("Improved Results:\n", improved_results)

----- Enhancement Step: Improving Summary ----------------------------


Output()


Improved Summary:
 In July 2025, Project NANDA released a report detailing the 'GenAI Divide,' a phenomenon indicating that, despite substantial enterprise investments of $30 billion to $40 billion in generative AI, around 95% of organizations have seen negligible financial returns on these implementations. The study distinguishes between 'builders'—those creating AI technologies—and 'buyers'—those implementing them. Although over 80% of organizations have engaged with widely available AI tools like ChatGPT, few are realizing significant value from tailored AI systems. The report identifies several critical barriers to transformation, including misalignment with daily workflows and inadequate feedback mechanisms in existing AI solutions. Key patterns contributing to the GenAI Divide include: limited disruptive change across most industries, larger firms leading in pilot initiatives but failing to scale effectively, an investment focus biased towards visible results over substantial RO

Output()

Output()

Output()

Improved Results:
 {'SummarizationScore': 0.23529411764705882, 'CoherenceScore': 0.9, 'TonalityScore': 0.9, 'SafetyScore': 0.8}


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
